In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import numpy as np
import glob
from matplotlib.colors import LinearSegmentedColormap

## Handle files

### split in 50/50 and combine duplicates

In [ ]:
# Assuming your data is in a DataFrame called df
df = pd.read_csv('/Path_file/Counts_TRAV_TRAJ_pairs_prod_AllIGTs.csv', sep='\t')  # Uncomment this line if reading from a file

# Load the file containing the position data
positions_df = pd.read_csv('/Path_file/TRAV_TRAJ_positions.csv')  # Uncomment this line if reading from a file



# Function to split TRAV and adjust counts and frequencies
def split_and_adjust(df):
    new_rows = []
    for _, row in df.iterrows():
        travs = row['TRAV'].split('|')
        if len(travs) == 1:
            new_rows.append(row)
        else:
            count = row['Count'] / 2
            frequency = row['Frequency'] / 2
            for trav in travs:
                new_row = row.copy()
                new_row['TRAV'] = trav
                new_row['Count'] = count
                new_row['Frequency'] = frequency
                new_rows.append(new_row)

    new_df = pd.DataFrame(new_rows)

    # Group by TRAV and TRAJ, then sum the Count and Frequency
    combined_df = new_df.groupby(['TRAV', 'TRAJ'], as_index=False).agg({
        'Count': 'sum',
        'Frequency': 'sum'
    })

    return combined_df

# Apply the function
combined_df = split_and_adjust(df)

# Merge with the positions DataFrame to add Position TRAV and Position TRAJ
result_df = pd.merge(
    combined_df,
    positions_df[['TRAV', 'TRAJ', 'Position TRAV', 'Position TRAJ']],
    on=['TRAV', 'TRAJ'],
    how='left'
)

# Save the result to a new CSV file
result_df.to_csv('/Path_output/Counts_TRAV_TRAJ_pairs_prod_AllIGTs_50_50.csv', index=False)

# Display the result
print(result_df)

## Heatmap positions genes

### Plot all data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# Load the final filtered CSV file into a DataFrame
data = pd.read_csv('Path_output/Counts_TRAV_TRAJ_pairs_prod_AllIGTs_50_50.csv', sep=',')
df = pd.DataFrame(data)

# Sort TRAV and TRAJ genes by their genomic positions
sorted_trav = df.sort_values(by="Position TRAV")["TRAV"].unique()
sorted_traj = df.sort_values(by="Position TRAJ")["TRAJ"].unique()

# Create a frequency matrix (initialize with zeros)
freq_matrix = pd.DataFrame(0, index=sorted_trav, columns=sorted_traj, dtype=float)

# Fill the matrix with frequencies from your data
for _, row in df.iterrows():
    trav, traj, freq = row["TRAV"], row["TRAJ"], row["Frequency"]
    freq_matrix.loc[trav, traj] = freq

# Remove "TRAV" and "TRAJ" from the labels
freq_matrix.index = freq_matrix.index.str.replace("TRAV", "").str.replace("TRAJ", "")
freq_matrix.columns = freq_matrix.columns.str.replace("TRAV", "").str.replace("TRAJ", "")

# Plot
plt.figure(figsize=(8, 9))

# === Define two colormaps ===
cmap = LinearSegmentedColormap.from_list("white_red", ["white", "red"])


# Highlight heatmap (white → red)
sns.heatmap(
    freq_matrix,
    cmap=cmap,
    cbar_kws={
        'label': 'Rearrangement Frequency (%)',
        'fraction': 0.02,  # Reduces the height of the color bar
        'shrink': 0.5,     # Scales the color bar width
        'pad': 0.02        # Adjusts padding around the color bar
    }
)

plt.title("Frequency of VaJa Joins")
plt.xlabel("TRAJ genes (5'→3')", fontsize=12)
plt.ylabel("TRAV genes (5'→3')", fontsize=12)
#plt.xticks(rotation=90, fontsize=6)
plt.xticks(ticks=np.arange(len(freq_matrix.columns)) + 0.5, labels=freq_matrix.columns, rotation=90, fontsize=6, ha='center')
#plt.yticks(rotation=0, fontsize=7)
plt.yticks(ticks=np.arange(len(freq_matrix.index)) + 0.5, labels=freq_matrix.index, rotation=0, fontsize=6, va='center')
plt.tight_layout()

output_path = 'Path_output/VaJa_pairing_positions_allIGTs_heatmap.svg'
plt.savefig(output_path, dpi=500, format='svg')
plt.show()
